# TP Python 2 : GeoPandas — Données vectorielles

**Géoinformatique II — Université de Lausanne**

---

## Objectifs pédagogiques

À la fin de ce TP, tu seras capable de :

1. Comprendre la structure d'un `GeoDataFrame` (géométries, attributs, CRS)
2. Effectuer des requêtes attributaires avec `.loc`, `.iloc` et `.query()`
3. Réaliser des jointures attributaires (`merge`) et spatiales (`sjoin`)
4. Reprojeter des données géographiques avec `.to_crs()`
5. Créer des cartes choroplèthes personnalisées avec Matplotlib
6. Exporter des données en shapefile et GeoJSON

> 💡 **Prérequis** : Ce TP fait suite au TP Python 1. Assure-toi d'avoir GeoPandas installé.


In [ ]:
# Importation des bibliothèques nécessaires
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

print(f"GeoPandas version : {gpd.__version__}")
print(f"Pandas version    : {pd.__version__}")


---
## 1. GeoDataFrame — structure et géométries

Un `GeoDataFrame` est un tableau de données (comme un `DataFrame` Pandas)
auquel s'ajoute une colonne `geometry` contenant les géométries des entités.

### 1.1 Les types de géométries

| Type | Description | Exemple SIG |
|---|---|---|
| `Point` | Un emplacement (x, y) | Ville, station météo |
| `LineString` | Une séquence de points reliés | Route, rivière |
| `Polygon` | Une surface fermée | Commune, canton, pays |
| `MultiPolygon` | Plusieurs polygones pour une entité | Pays insulaires |


In [ ]:
# Création d'un GeoDataFrame à partir de zéro
# (en pratique, on lit des fichiers, mais c'est utile pour comprendre la structure)
from shapely.geometry import Point, LineString, Polygon

# Quelques villes suisses avec leurs coordonnées WGS84
data = {
    'ville': ['Lausanne', 'Genève', 'Berne', 'Zurich', 'Bâle', 'Lugano'],
    'population': [139408, 203856, 133883, 428745, 179053, 63932],
    'altitude_m': [496, 375, 542, 408, 261, 273],
    'geometry': [
        Point(6.6323, 46.5197),
        Point(6.1432, 46.2044),
        Point(7.4458, 46.9480),
        Point(8.5417, 47.3769),
        Point(7.5886, 47.5596),
        Point(8.9530, 46.0037),
    ]
}

villes = gpd.GeoDataFrame(data, crs='EPSG:4326')   # WGS84

print(villes)
print()
print(f"CRS   : {villes.crs}")
print(f"Forme : {villes.shape}")
print(f"Types de géométrie : {villes.geom_type.unique()}")


In [ ]:
# --- Accès aux composantes d'une géométrie ---
lausanne_geom = villes.loc[0, 'geometry']    # objet Shapely Point
print(f"Géométrie : {lausanne_geom}")
print(f"Type      : {lausanne_geom.geom_type}")
print(f"Longitude : {lausanne_geom.x}")
print(f"Latitude  : {lausanne_geom.y}")

# Bounding box (emprise) du jeu de données entier
print(f"\nEmprise (bounds) : {villes.total_bounds}")
# [xmin, ymin, xmax, ymax]


In [ ]:
# --- Afficher les villes ---
fig, ax = plt.subplots(figsize=(7, 7))

# Charge le fond de carte (pays du monde)
monde = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
suisse = monde[monde['name'] == 'Switzerland']

# Affiche la Suisse
suisse.plot(ax=ax, color='#f5f5f5', edgecolor='#333', linewidth=1.5)

# Affiche les villes avec taille proportionnelle à la population
villes.plot(ax=ax, markersize=villes['population'] / 2000,
            color='steelblue', alpha=0.8, zorder=5)

# Étiquettes
for _, row in villes.iterrows():
    ax.annotate(row['ville'], xy=(row.geometry.x, row.geometry.y),
                xytext=(4, 4), textcoords='offset points', fontsize=9)

ax.set_title('Principales villes de Suisse', fontsize=13)
ax.set_xlabel('Longitude (°)')
ax.set_ylabel('Latitude (°)')
plt.tight_layout()
plt.show()


### 1.2 Chargement de données réelles

En pratique, on lit le plus souvent des fichiers existants :
- **Shapefile** (`.shp`) : format propriétaire Esri, très répandu
- **GeoJSON** (`.geojson`) : format texte ouvert, idéal pour le web
- **GeoPackage** (`.gpkg`) : format ouvert et moderne recommandé par l'OGC

Les deux méthodes ci-dessous donnent le même résultat :
```python
gdf = gpd.read_file("mon_fichier.shp")
gdf = gpd.read_file("http://exemple.com/donnees.geojson")
```


In [ ]:
# Chargement du jeu de données monde avec ses attributs complets
monde = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))

print("Colonnes disponibles :")
print(monde.columns.tolist())
print()
print("5 premières lignes :")
monde.head()


---
## 2. Requêtes attributaires

La sélection et le filtrage des données est l'une des opérations les plus
fréquentes en géomatique. GeoPandas hérite des mécanismes de Pandas.

### 2.1 Sélection avec `.loc` et `.iloc`

- `.loc[lignes, colonnes]` : sélection par **étiquettes** (noms de colonnes, index)
- `.iloc[lignes, colonnes]` : sélection par **position** (numéros entiers)


In [ ]:
# .iloc — sélection par position
print("Première ligne (iloc[0]) :")
print(monde.iloc[0])
print()
print("Lignes 0 à 4, colonnes 0 à 3 :")
print(monde.iloc[0:5, 0:4])


In [ ]:
# .loc — sélection par étiquettes
print("Colonnes 'name' et 'pop_est' pour les 5 premières lignes :")
print(monde.loc[0:4, ['name', 'pop_est']])


In [ ]:
# Sélection par condition booléenne
# (la plus courante en pratique)

# Pays d'Afrique subsaharienne
afrique_sub = monde[monde['continent'] == 'Africa']

# Pays avec une population > 50 millions
grands_pays = monde[monde['pop_est'] > 50_000_000]

# Pays européens avec PIB estimé > 500 milliards USD
europe_riche = monde[
    (monde['continent'] == 'Europe') &
    (monde['gdp_md_est'] > 500_000)
]

print(f"Pays africains          : {len(afrique_sub)}")
print(f"Pays > 50 M hab.        : {len(grands_pays)}")
print(f"Pays europe riches      : {len(europe_riche)}")
print(europe_riche[['name', 'pop_est', 'gdp_md_est']])


### 2.2 Requêtes avec `.query()`

La méthode `.query()` accepte une expression sous forme de chaîne de caractères,
ce qui la rend plus lisible pour des conditions complexes.


In [ ]:
# Équivalents avec .query()
afrique_sub_q = monde.query("continent == 'Africa'")
grands_pays_q = monde.query("pop_est > 50_000_000")
europe_riche_q = monde.query("continent == 'Europe' and gdp_md_est > 500_000")

# Variable externe dans une query (préfixe @)
seuil_pop = 20_000_000
pays_moyens = monde.query("pop_est > @seuil_pop and pop_est < 50_000_000")
print(f"Pays entre 20 M et 50 M hab. : {len(pays_moyens)}")
print(pays_moyens[['name', 'continent', 'pop_est']].sort_values('pop_est', ascending=False))


> 🎯 **Exercice 2** : En utilisant le `GeoDataFrame` `monde` :
> 1. Sélectionne tous les pays d'Océanie. Combien y en a-t-il ?
> 2. Trouve le pays d'Amérique du Sud avec la plus petite superficie
>    (utilise la colonne `geometry` et la méthode `.area` — attention au CRS !).
> 3. Avec `.query()`, liste les pays dont le nom contient la lettre "z"
>    (utilise l'expression `name.str.contains('z', case=False)`).


---
## 3. Jointures

Les jointures permettent de **combiner deux tables** en fonction d'un attribut commun
(jointure attributaire) ou d'une relation spatiale (jointure spatiale).

### 3.1 Jointure attributaire — `merge()`

Similaire à un SQL JOIN ou à la jointure de table dans QGIS.


In [ ]:
# --- Création d'une table de données supplémentaires ---
# Indice de développement humain (IDH) pour quelques pays européens
# (données fictives pour l'exemple)
idh_data = pd.DataFrame({
    'pays': ['France', 'Germany', 'Italy', 'Spain', 'Switzerland',
             'Austria', 'Belgium', 'Netherlands', 'Portugal', 'Sweden'],
    'idh_2023': [0.903, 0.942, 0.895, 0.905, 0.962,
                 0.916, 0.919, 0.941, 0.866, 0.952],
    'esperance_vie': [82.3, 81.7, 83.4, 83.5, 84.2,
                      81.9, 81.6, 82.3, 81.4, 83.1]
})

print("Table IDH :")
print(idh_data)


In [ ]:
# --- Jointure attributaire ---
# On joint le GeoDataFrame 'monde' avec la table 'idh_data'
# La colonne de jointure est 'name' dans monde et 'pays' dans idh_data
europe = monde[monde['continent'] == 'Europe'].copy()

europe_idh = europe.merge(
    idh_data,
    left_on='name',     # colonne de clé dans europe
    right_on='pays',    # colonne de clé dans idh_data
    how='left'          # garde tous les pays européens, même sans IDH
)

print(f"\nRésultat de la jointure : {europe_idh.shape}")
print(europe_idh[['name', 'pop_est', 'idh_2023', 'esperance_vie']].dropna().head(10))


### 3.2 Jointure spatiale — `sjoin()`

La jointure spatiale associe des entités en fonction de leur **relation
géométrique** (intersection, contenance, proximité).

> 🔗 **Parallèle QGIS** : outil *Joindre les attributs par localisation* ou
> *Sélection par localisation*.


In [ ]:
# --- Exemple de jointure spatiale ---
# On veut savoir dans quel pays se trouve chaque ville de notre liste

# Notre GeoDataFrame de villes (déjà créé plus haut)
print("Villes :")
print(villes[['ville', 'geometry']])
print()

# GeoDataFrame des pays
pays = monde[['name', 'continent', 'pop_est', 'geometry']].copy()

# sjoin : pour chaque ville, on cherche le polygone pays qui la contient
villes_dans_pays = gpd.sjoin(
    villes,                         # GeoDataFrame gauche (points)
    pays,                           # GeoDataFrame droit (polygones)
    how='left',                     # garde toutes les villes
    predicate='within'              # relation spatiale : point dans polygone
)

print("Résultat de la jointure spatiale :")
print(villes_dans_pays[['ville', 'name', 'continent', 'population']])


> 🎯 **Exercice 3** : Crée un nouveau `GeoDataFrame` contenant 5 capitales mondiales
> de ton choix (avec leur géométrie Point en WGS84), puis utilise `sjoin` pour
> trouver dans quel continent se trouve chaque capitale.


---
## 4. Reprojection — `.to_crs()`

Le **CRS (Coordinate Reference System)** définit comment les coordonnées
sont ancrées sur la Terre. Il est crucial de s'assurer que toutes les
couches utilisées partagent le même CRS.

| CRS | Code EPSG | Utilisation |
|---|---|---|
| WGS84 (géographique) | 4326 | GPS, données mondiales |
| Swiss LV95 / MN95 | 2056 | Données suisses (swisstopo) |
| Web Mercator | 3857 | Tuiles web, OpenStreetMap |
| UTM 32N | 32632 | Europe centrale (mesures métriques) |

> ⚠️ **Important** : `.area`, `.length` et `.distance()` retournent des valeurs
> dans l'unité du CRS. En WGS84 (degrés), ces valeurs n'ont pas de sens
> physique direct — il faut d'abord reprojeter en système métrique.


In [ ]:
# --- Impact du CRS sur les mesures ---
monde = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
france = monde[monde['name'] == 'France'].copy()

# Superficie en WGS84 (degrés² — peu interprétable !)
area_wgs84 = france.geometry.area.values[0]
print(f"Superficie WGS84 (degrés²)    : {area_wgs84:.4f}")

# Reprojection en UTM 32N (système métrique, bon pour la France)
france_utm = france.to_crs(epsg=32632)
area_utm_km2 = france_utm.geometry.area.values[0] / 1e6
print(f"Superficie UTM 32N (km²)      : {area_utm_km2:,.0f} km²")
print(f"(Superficie réelle France ≈ 551 695 km²)")


In [ ]:
# --- Reprojection de notre couche de villes suisses ---
print("CRS initial :", villes.crs)

# Reprojection en MN95 (Swiss LV95) — système officiel suisse
villes_mn95 = villes.to_crs(epsg=2056)
print("CRS après reprojection :", villes_mn95.crs)
print()

# Les coordonnées ont changé !
print("Coordonnées WGS84  (Lausanne) :",
      villes.loc[0, 'geometry'].x, villes.loc[0, 'geometry'].y)
print("Coordonnées MN95   (Lausanne) :",
      round(villes_mn95.loc[0, 'geometry'].x),
      round(villes_mn95.loc[0, 'geometry'].y))


In [ ]:
# --- Calcul de distances après reprojection ---
# Calcul de la distance entre Lausanne et Zurich (en km)
villes_mn95 = villes.to_crs(epsg=2056)

lausanne_pt = villes_mn95[villes_mn95['ville'] == 'Lausanne'].geometry.values[0]
zurich_pt   = villes_mn95[villes_mn95['ville'] == 'Zurich'].geometry.values[0]

distance_m = lausanne_pt.distance(zurich_pt)
print(f"Distance Lausanne–Zurich (MN95) : {distance_m / 1000:.1f} km")
print("(Distance réelle à vol d'oiseau : ≈ 175 km)")


> 🎯 **Exercice 4** : 
> 1. Charge le jeu de données `naturalearth_lowres`, isole l'Amérique du Sud.
> 2. Reprojette ce GeoDataFrame en EPSG:5641 (système équivalent-aréal Amérique du Sud).
> 3. Calcule la superficie de chaque pays en km² et trie-les du plus grand au plus petit.
> 4. Quel est le plus grand pays d'Amérique du Sud selon ce calcul ?


---
## 5. Cartographie choroplèthe

Une **carte choroplèthe** colore les entités géographiques en fonction de la
valeur d'un attribut numérique. C'est l'un des types de cartes thématiques
les plus courants en géomatique.

### 5.1 Carte choroplèthe simple avec GeoPandas


In [ ]:
monde = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))

fig, ax = plt.subplots(figsize=(14, 7))

monde.plot(
    ax=ax,
    column='pop_est',          # attribut à visualiser
    cmap='YlOrRd',             # palette de couleurs
    scheme='quantiles',        # méthode de discrétisation
    k=5,                       # nombre de classes
    legend=True,
    legend_kwds={
        'title': 'Population',
        'loc': 'lower left',
    },
    edgecolor='white',
    linewidth=0.3,
    missing_kwds={'color': 'lightgrey'}
)

ax.set_title('Population mondiale par pays (2019)', fontsize=15, pad=10)
ax.set_facecolor('#c7e9f1')    # couleur des océans
ax.set_axis_off()              # masque les axes
plt.tight_layout()
plt.show()


### 5.2 Méthodes de discrétisation

Le choix de la **méthode de classification** est crucial : elle influence
fortement la perception de la carte.

| Méthode (`scheme=`) | Description |
|---|---|
| `'quantiles'` | Classes avec le même nombre d'entités |
| `'equal_interval'` | Classes avec le même intervalle de valeurs |
| `'natural_breaks'` (Jenks) | Minimise la variance intra-classe |
| `'fisher_jenks'` | Variante plus stable de Jenks |
| `'std_mean'` | Classes basées sur l'écart-type |


In [ ]:
# --- Comparaison de méthodes de discrétisation ---
europe = monde[monde['continent'] == 'Europe'].copy()
europe_proj = europe.to_crs(epsg=3035)   # LAEA Europe — mètres

# Calcul densité de population (hab/km²)
europe_proj['superficie_km2'] = europe_proj.geometry.area / 1e6
europe_proj['densite'] = europe_proj['pop_est'] / europe_proj['superficie_km2']

methodes = ['quantiles', 'equal_interval', 'natural_breaks']
titres   = ['Quantiles (5 classes)', 'Intervalles égaux (5 classes)', 'Jenks (5 classes)']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, methode, titre in zip(axes, methodes, titres):
    europe_proj.plot(
        ax=ax, column='densite',
        cmap='OrRd', scheme=methode, k=5,
        legend=True, legend_kwds={'fontsize': 7},
        edgecolor='white', linewidth=0.4,
        missing_kwds={'color': 'lightgrey'}
    )
    ax.set_title(titre, fontsize=10)
    ax.set_axis_off()

plt.suptitle('Densité de population en Europe — méthodes de discrétisation', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# --- Carte choroplèthe avec éléments cartographiques ---
europe = monde[monde['continent'] == 'Europe'].to_crs(epsg=3035).copy()
europe['densite'] = europe['pop_est'] / (europe.geometry.area / 1e6)

fig, ax = plt.subplots(figsize=(10, 10))

# Fond de carte
europe.plot(ax=ax, color='#f0f0f0', edgecolor='#aaa', linewidth=0.5)

# Choroplèthe
europe.plot(
    ax=ax,
    column='densite',
    cmap='Oranges',
    scheme='natural_breaks', k=5,
    legend=True,
    legend_kwds={
        'title': 'Densité
(hab/km²)',
        'loc': 'lower right',
        'fontsize': 9,
        'title_fontsize': 10,
    },
    edgecolor='white', linewidth=0.4,
    missing_kwds={'color': '#cccccc', 'label': 'Données manquantes'}
)

# Étiquettes pour les grands pays
for _, row in europe.iterrows():
    if row['pop_est'] > 10_000_000:
        ax.annotate(row['name'],
                    xy=(row.geometry.centroid.x, row.geometry.centroid.y),
                    ha='center', fontsize=7, color='#333',
                    bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.5, lw=0))

ax.set_title('Densité de population en Europe', fontsize=14, pad=12)
ax.set_axis_off()

# Note de source
ax.annotate('Source : Natural Earth, 2019',
            xy=(0.01, 0.01), xycoords='axes fraction',
            fontsize=8, color='grey')

plt.tight_layout()
plt.show()


> 🎯 **Exercice 5** : Crée une carte choroplèthe de l'**Asie** représentant
> le **PIB estimé par habitant** (`gdp_md_est` / `pop_est` × 1 000 000).
> - Reprojette en EPSG:3857 (Web Mercator) ou un CRS adapté à l'Asie
> - Utilise 5 classes avec la méthode `'quantiles'`
> - Choisir une palette de couleur appropriée (ex. `'YlGnBu'`)
> - Ajouter un titre et masquer les axes


---
## 6. Export — shapefile et GeoJSON

Une fois les données traitées, il faut souvent les **sauvegarder** dans un
format standard pour les partager ou les ouvrir dans un autre logiciel.


In [ ]:
import os

# Création d'un dossier de sortie
os.makedirs("sortie", exist_ok=True)

# --- Export en Shapefile ---
# Attention : les shapefiles tronquent les noms de colonnes à 10 caractères
# et ne supportent pas les caractères spéciaux dans les attributs.
europe = monde[monde['continent'] == 'Europe'].copy()
europe.to_file("sortie/europe.shp")
print("Shapefile sauvegardé : sortie/europe.shp")

# --- Export en GeoJSON ---
# GeoJSON est plus pratique : noms de colonnes sans limite, un seul fichier
europe.to_file("sortie/europe.geojson", driver='GeoJSON')
print("GeoJSON sauvegardé  : sortie/europe.geojson")

# --- Export en GeoPackage (recommandé) ---
# Format ouvert, un seul fichier, supporte plusieurs couches
europe.to_file("sortie/europe.gpkg", layer='europe', driver='GPKG')
villes.to_file("sortie/europe.gpkg", layer='villes_suisse', driver='GPKG')
print("GeoPackage sauvegardé : sortie/europe.gpkg (2 couches)")


In [ ]:
# --- Vérification : relire un fichier exporté ---
europe_relu = gpd.read_file("sortie/europe.geojson")
print(f"Fichier relu : {europe_relu.shape[0]} entités, CRS = {europe_relu.crs}")

# --- Export d'une sélection avec reprojection ---
# Exemple : exporter les pays européens en MN95 (pour usage avec données suisses)
europe_mn95 = europe.to_crs(epsg=2056)
europe_mn95.to_file("sortie/europe_mn95.gpkg", driver='GPKG')
print("Export MN95 sauvegardé.")


> 🎯 **Exercice 6 — projet intégrateur** :
> En combinant les techniques vues dans ce TP, réalise le workflow suivant :
>
> 1. Charge le jeu de données `naturalearth_lowres`
> 2. Isole les pays d'**Amérique du Nord**
> 3. Reprojette en **EPSG:5070** (Albers Equal Area — Amérique du Nord)
> 4. Calcule la densité de population en hab/km²
> 5. Crée une carte choroplèthe de la densité avec 5 classes (méthode : `natural_breaks`)
> 6. Exporte le résultat en GeoJSON dans le dossier `sortie/`


---
## Récapitulatif

Dans ce TP, tu as appris à :

- ✅ Comprendre la structure d'un `GeoDataFrame` (géométries, attributs, CRS)
- ✅ Créer des géométries Shapely et construire un `GeoDataFrame` depuis zéro
- ✅ Sélectionner et filtrer des données avec `.loc`, `.iloc` et `.query()`
- ✅ Réaliser des jointures attributaires (`merge`) et spatiales (`sjoin`)
- ✅ Reprojeter des données et comprendre l'impact sur les mesures
- ✅ Produire des cartes choroplèthes personnalisées avec plusieurs méthodes de discrétisation
- ✅ Exporter des GeoDataFrames en shapefile, GeoJSON et GeoPackage

### Ressources pour aller plus loin

- [Documentation GeoPandas](https://geopandas.org/en/stable/docs.html)
- [Guide Shapely](https://shapely.readthedocs.io/)
- [Natural Earth Data](https://www.naturalearthdata.com/)
- [Données swisstopo Open Data](https://www.swisstopo.admin.ch/fr/geodata-products/geodata)

**Prochain TP** : Analyse raster avec Rasterio et NumPy.
